In [ ]:
import pandas as pd
import joblib

def diagnose_patient(age, gender, t3, ft4, tsh, anti_tpo_ordered, anti_tg_ordered):
    """
    Takes raw patient lab results and predicts their clinic using the Hierarchical AI.
    """
    print("\n--- Processing Patient Data ---")
    
    try:
        stage1_model = joblib.load('models/hierarchical_stage1.pkl')
        stage2_model = joblib.load('models/hierarchical_stage2.pkl')
    except FileNotFoundError:
        return "Error: Could not find the models. Did you run the save code in Notebook 08?"


    gender_enc = 1 if gender.lower() in ['m', 'male'] else 0

    t3_ft4_ratio = t3 / (ft4 + 0.001)
    t3_tsh_ratio = t3 / (tsh + 0.001)
    
    tt4ri = ft4 * tsh
    deiodinase_proxy = t3_ft4_ratio * (1 - (age / 100))
    expected_tsh = 1.5 + (0.025 * age)
    tsh_deviation = tsh - expected_tsh

    # 3. Format the data exactly how the AI expects it
    feature_cols = [
        'T3', 'FT4', 'TSH', 'Age', 'Gender_Encoded',
        'T3_FT4_ratio', 'T3_TSH_ratio',
        'AntiTPO_Measured', 'AntiTG_Measured',
        'TT4RI', 'Deiodinase_Proxy', 'TSH_Deviation'
    ]

    patient_df = pd.DataFrame([[
        t3, ft4, tsh, age, gender_enc,
        t3_ft4_ratio, t3_tsh_ratio,
        int(anti_tpo_ordered), int(anti_tg_ordered),
        tt4ri, deiodinase_proxy, tsh_deviation
    ]], columns=feature_cols)


    print("Running Stage 1: General vs. Specialized...")
    is_specialized = stage1_model.predict(patient_df)[0] 

    if is_specialized == 0:
        print(" Result: Standard Outpatient detected.")
        return "Aimolipsies (General Blood Draw)"
    
    else:
        print(" Result: Severe Systemic Illness (NTIS) detected. Moving to Stage 2...")
        
        specialized_pred = stage2_model.predict(patient_df)[0]
        
        if specialized_pred == 0:
            return "Kardiologiki (Cardiology Ward)"
        else:
            return "Nefrologiki (Nephrology Ward)"

if __name__ == "__main__":
    
    prediction = diagnose_patient(
        age=27, 
        gender='M', 
        t3=1.03,       
        ft4=1.34,      
        tsh=1.140,      
        anti_tpo_ordered=True, 
        anti_tg_ordered=True
    )
    
    print("\n==================================")
    print(f" AI PREDICTION: {prediction}")
    print("==================================\n")


--- Processing Patient Data ---
Running Stage 1: General vs. Specialized...
 Result: Standard Outpatient detected.

 AI PREDICTION: Aimolipsies (General Blood Draw)

